# Administrative Data Removal

This notebook implements automated administrative field removal for QC anomaly modelling datasets.

The preprocessing stage removes non-predictive identifier-related columns to reduce data leakage and prevent future anomaly detection models from memorising operational metadata instead of learning meaningful chemical and QC-related patterns.

In [61]:
import pandas as pd

In [62]:
df = pd.read_csv(
    "QC_Sample_Data.csv",
    dtype={"INSTRUMENT_ID": "string"}
)

In [63]:
# Inspect Dataset Columns

#The dataset structure is reviewed to identify administrative and analytical variables.

In [64]:
df.shape

df.columns

Index(['ANALYTICAL_TYPE', 'SAMPLE_NAME_ANON', 'JOB_NAME_ANON',
       'NUMERIC_FINAL_VALUE', 'INTERNAL_TARGET_VALUE', 'ANALYSED_DATE',
       'RECOVERY', 'SCHEME_CODE', 'SCHEME_VERSION', 'ANALYTE_CODE',
       'RELATIVE_PERCENT_DIFFERENCE', 'STANDARD_STATUS', 'RECOVERY.1',
       'PRECISION_STATUS', 'ANALYSED_BY_USER_ID', 'JOB_CLIENT_ID', 'UNIT_CODE',
       'INSTRUMENT_CODE'],
      dtype='str')

# Automated Administrative Field Detection

The preprocessing logic automatically identifies identifier-related fields using keyword-based matching rules.

This approach allows future datasets to be processed automatically without requiring manual column selection.

In [65]:
admin_keywords = [
    "JOB",
    "USER",
    "CLIENT",
    "NAME",
    "ID",
    "INSTRUMENT",
    "ANON"
]

protected_columns = [
    "ANALYTE_CODE",
    "SCHEME_CODE",
    "STANDARD_STATUS",
    "PRECISION_STATUS"
]

In [66]:
# Administrative keywords commonly associated
# with identifiers or operational metadata
admin_keywords = [
    "JOB",
    "USER",
    "CLIENT",
    "OPERATOR",
    "INSTRUMENT",
    "SESSION",
    "RUN"
]

# Scientific / QC fields that should NEVER
# be removed even if they contain ID-like text
protected_columns = [
    "ANALYTE_CODE",
    "SCHEME_CODE",
    "STD_CODE",
    "STD_LOT_CODE",
    "SPECIFICATION_CODE"
]


# Function to automatically remove
# administrative and identifier-related fields
def remove_admin_fields(df):

    # Store detected administrative columns
    cols_to_remove = []

    # Loop through dataset columns
    for col in df.columns:

        # Convert column name to uppercase
        # for case-insensitive matching
        col_upper = col.upper()

        # Check if column contains
        # administrative keywords
        if any(keyword in col_upper for keyword in admin_keywords):

            # Prevent important scientific
            # QC variables from being removed
            if col not in protected_columns:

                cols_to_remove.append(col)

        # Additional logic:
        # Remove extremely high-uniqueness columns
        # which are likely identifiers
        elif df[col].nunique() == len(df):

            # Protect important scientific variables
            if col not in protected_columns:

                cols_to_remove.append(col)

    # Remove detected columns
    cleaned_df = df.drop(
        columns=cols_to_remove,
        errors="ignore"
    )

    # Return cleaned dataset
    # and removed column names
    return cleaned_df, cols_to_remove


# Run preprocessing
clean_df, removed_cols = remove_admin_fields(df)


In [67]:
print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)

print("Removed columns:")
print(removed_cols)

print("Remaining columns:")
print(clean_df.columns)

Original shape: (1201152, 18)
Cleaned shape: (1201152, 14)
Removed columns:
['JOB_NAME_ANON', 'ANALYSED_BY_USER_ID', 'JOB_CLIENT_ID', 'INSTRUMENT_CODE']
Remaining columns:
Index(['ANALYTICAL_TYPE', 'SAMPLE_NAME_ANON', 'NUMERIC_FINAL_VALUE',
       'INTERNAL_TARGET_VALUE', 'ANALYSED_DATE', 'RECOVERY', 'SCHEME_CODE',
       'SCHEME_VERSION', 'ANALYTE_CODE', 'RELATIVE_PERCENT_DIFFERENCE',
       'STANDARD_STATUS', 'RECOVERY.1', 'PRECISION_STATUS', 'UNIT_CODE'],
      dtype='str')


Detection Limit Numeric Conversion



To Do


Description

Apply the half-limit rule to all samples. This converts text strings representing values below the detection limit into a numeric noise floor, allowing mathematical comparisons.
# Detection Limit Numeric Conversion

Laboratory QC datasets may contain measurements recorded below the analytical detection limit using text-based representations such as:

- <0.01
- <5
- <100

These values cannot be directly used in machine learning or statistical calculations because they are stored as text rather than numeric values.

To address this, the half-limit rule was implemented to automatically convert below-detection-limit measurements into small numeric approximations while preserving their interpretation as low-concentration observations.

In [68]:
# Count values containing "<"

df["NUMERIC_FINAL_VALUE"] \
    .astype(str) \
    .str.contains("<") \
    .sum()

np.int64(0)

# Detection Limit Investigation Outcome

No below-detection-limit textual measurements (e.g. <0.01 or <5) were identified within the current historical QC dataset.

This suggests that the exported historical data has already undergone preprocessing or numeric standardisation prior to extraction from the laboratory system.

However, an automated half-limit conversion function was still implemented to support future raw batch-level datasets where below-detection-limit values may appear during real-time QC processing.

In [69]:
# Function to convert below-detection-limit text values
# into numeric half-limit approximations

def apply_half_limit_rule(value):

    # Convert value to string for safe text processing
    value_str = str(value).strip()

    # Check whether value starts with "<"
    if value_str.startswith("<"):

        try:
            # Extract numeric portion after "<"
            limit_value = float(value_str.replace("<", ""))

            # Apply half-limit rule
            return limit_value / 2

        except:
            # Return NaN if conversion fails
            return np.nan

    # Otherwise return original value
    return value

In [70]:
# Create a copy of the dataset
converted_df = df.copy()

# Apply half-limit conversion to numeric result column
converted_df["NUMERIC_FINAL_VALUE"] = (
    converted_df["NUMERIC_FINAL_VALUE"]
    .apply(apply_half_limit_rule)
)

# Convert column into numeric datatype
converted_df["NUMERIC_FINAL_VALUE"] = pd.to_numeric(
    converted_df["NUMERIC_FINAL_VALUE"],
    errors="coerce"
)

# Display datatype after conversion
print(converted_df["NUMERIC_FINAL_VALUE"].dtype)

# Display sample values
converted_df["NUMERIC_FINAL_VALUE"].head()

float64


0    0.633691
1    1.340977
2    0.616357
3    3.949210
4    0.623754
Name: NUMERIC_FINAL_VALUE, dtype: float64

# Meaning

The historical QC dataset has already been converted into numeric form.

So:
- there are no remaining textual values like <0.01,
- the exported historical data is already machine-learning compatible.

In [71]:
test_values = pd.Series([
    "<0.01",
    "<5",
    "10.5",
    "200"
])

test_values = test_values.apply(apply_half_limit_rule)

test_values = pd.to_numeric(
    test_values,
    errors="coerce"
)

print(test_values)
print(test_values.dtype)

0      0.005
1      2.500
2     10.500
3    200.000
dtype: float64
float64


# The value becomes:
- numeric,
- scalable,
- statistically comparable,
- usable for anomaly detection.

The preprocessing pipeline was validated using synthetic detection-limit examples to ensure future compatibility with raw laboratory batch data. The implemented half-limit logic successfully converted textual below-detection-limit measurements into numeric approximations while preserving standard numeric observations and producing machine-learning-compatible float values

Investigation of the QC history extraction query showed that the exported dataset already contained numeric-standardised result fields such as NUMERIC_FINAL_VALUE and PARENT_NUMERIC_FINAL_VALUE. As a result, no raw below-detection-limit text values (e.g. <0.01) were present within the historical dataset. However, an automated half-limit preprocessing function was still implemented and validated to support future raw batch-level datasets where textual detection-limit measurements may still occur.